# Bài tập lớn Khai phá dữ liệu — Bài toán Phân cụm
## Đề tài: Phân khúc khách hàng siêu thị để gợi ý khuyến mãi

**Nhóm 5 người** *(điền danh sách thành viên khi nộp bài)*

- **Bộ dữ liệu:** Wholesale Customers Data Set (UCI Machine Learning Repository, 440 dòng, 8 cột).
- **Tiền xử lý:** Kiểm tra dữ liệu, loại bỏ ngoại lai bằng quy tắc **IQR**, chuẩn hóa bằng **StandardScaler** và đối chiếu **MinMaxScaler**.
- **3 thuật toán so sánh:** **K-Means**, **Hierarchical Clustering (Agglomerative - Ward)** và **DBSCAN**.
- **Đánh giá:** **Elbow Method** (Khuỷu tay) và **Silhouette Score** (cùng Davies–Bouldin, Calinski–Harabasz).
- **Ứng dụng:** Giao diện Streamlit thu thập dữ liệu mới, tiền xử lý, huấn luyện lại định kỳ và hiển thị kết quả phân khúc + gợi ý khuyến mãi.

> **Cách chạy trên Google Colab:** Chọn *Runtime → Run all* (dùng CPU mặc định). Notebook tự tải dữ liệu từ UCI (có bản dự phòng nhúng sẵn nếu mạng lỗi) và có ô chạy trực tiếp giao diện Streamlit ở cuối.

In [ ]:
import io, sys, json, hashlib, shutil, zipfile, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display

print('Python', sys.version.split()[0], '| scikit-learn', sklearn.__version__, '| pandas', pd.__version__)
WORK = Path.cwd()
PROJECT_DIR = WORK / 'wholesale_app'
FIGURES = WORK / 'figures'
(PROJECT_DIR / 'data').mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(exist_ok=True)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .25})

## 1. Giới thiệu dữ liệu & Khám phá (EDA)
Nguồn: [UCI Wholesale Customers](https://archive.ics.uci.edu/dataset/292/wholesale+customers) — Cardoso, M. (2013), DOI `10.24432/C5030X`.

Bộ dữ liệu gồm **440 khách hàng**, mỗi dòng là tổng chi tiêu trong năm theo 6 nhóm mặt hàng:

| Thuộc tính | Ý nghĩa |
|---|---|
| `Fresh` | Chi tiêu nhóm hàng tươi sống |
| `Milk` | Chi tiêu sữa và chế phẩm |
| `Grocery` | Chi tiêu hàng tạp hóa |
| `Frozen` | Chi tiêu hàng đông lạnh |
| `Detergents_Paper` | Chi tiêu chất tẩy rửa & sản phẩm giấy |
| `Delicassen` | Chi tiêu thực phẩm chế biến sẵn |
| `Channel` | Kênh khách hàng: `1` = Nhà hàng/Khách sạn/Café; `2` = Bán lẻ / Siêu thị |
| `Region` | Khu vực: `1` = Lisbon; `2` = Oporto; `3` = Khu vực khác |

6 cột chi tiêu được dùng để tính khoảng cách phân cụm; `Channel` và `Region` dùng để mô tả và đối chiếu đặc điểm các cụm.

In [ ]:
EMBEDDED_CSV = 'Channel,Region,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen\n2,3,12669,9656,7561,214,2674,1338\n2,3,7057,9810,9568,1762,3293,1776\n2,3,6353,8808,7684,2405,3516,7844\n1,3,13265,1196,4221,6404,507,1788\n2,3,22615,5410,7198,3915,1777,5185\n2,3,9413,8259,5126,666,1795,1451\n2,3,12126,3199,6975,480,3140,545\n2,3,7579,4956,9426,1669,3321,2566\n1,3,5963,3648,6192,425,1716,750\n2,3,6006,11093,18881,1159,7425,2098\n2,3,3366,5403,12974,4400,5977,1744\n2,3,13146,1124,4523,1420,549,497\n2,3,31714,12319,11757,287,3881,2931\n2,3,21217,6208,14982,3095,6707,602\n2,3,24653,9465,12091,294,5058,2168\n1,3,10253,1114,3821,397,964,412\n2,3,1020,8816,12121,134,4508,1080\n1,3,5876,6157,2933,839,370,4478\n2,3,18601,6327,10099,2205,2767,3181\n1,3,7780,2495,9464,669,2518,501\n2,3,17546,4519,4602,1066,2259,2124\n1,3,5567,871,2010,3383,375,569\n1,3,31276,1917,4469,9408,2381,4334\n2,3,26373,36423,22019,5154,4337,16523\n2,3,22647,9776,13792,2915,4482,5778\n2,3,16165,4230,7595,201,4003,57\n1,3,9898,961,2861,3151,242,833\n1,3,14276,803,3045,485,100,518\n2,3,4113,20484,25957,1158,8604,5206\n1,3,43088,2100,2609,1200,1107,823\n1,3,18815,3610,11107,1148,2134,2963\n1,3,2612,4339,3133,2088,820,985\n1,3,21632,1318,2886,266,918,405\n1,3,29729,4786,7326,6130,361,1083\n1,3,1502,1979,2262,425,483,395\n2,3,688,5491,11091,833,4239,436\n1,3,29955,4362,5428,1729,862,4626\n2,3,15168,10556,12477,1920,6506,714\n2,3,4591,15729,16709,33,6956,433\n1,3,56159,555,902,10002,212,2916\n1,3,24025,4332,4757,9510,1145,5864\n1,3,19176,3065,5956,2033,2575,2802\n2,3,10850,7555,14961,188,6899,46\n2,3,630,11095,23998,787,9529,72\n2,3,9670,7027,10471,541,4618,65\n2,3,5181,22044,21531,1740,7353,4985\n2,3,3103,14069,21955,1668,6792,1452\n2,3,44466,54259,55571,7782,24171,6465\n2,3,11519,6152,10868,584,5121,1476\n2,3,4967,21412,28921,1798,13583,1163\n1,3,6269,1095,1980,3860,609,2162\n1,3,3347,4051,6996,239,1538,301\n2,3,40721,3916,5876,532,2587,1278\n2,3,491,10473,11532,744,5611,224\n1,3,27329,1449,1947,2436,204,1333\n1,3,5264,3683,5005,1057,2024,1130\n2,3,4098,29892,26866,2616,17740,1340\n2,3,5417,9933,10487,38,7572,1282\n1,3,13779,1970,1648,596,227,436\n1,3,6137,5360,8040,129,3084,1603\n2,3,8590,3045,7854,96,4095,225\n2,3,35942,38369,59598,3254,26701,2017\n2,3,7823,6245,6544,4154,4074,964\n2,3,9396,11601,15775,2896,7677,1295\n1,3,4760,1227,3250,3724,1247,1145\n2,3,85,20959,45828,36,24231,1423\n1,3,9,1534,7417,175,3468,27\n2,3,19913,6759,13462,1256,5141,834\n1,3,2446,7260,3993,5870,788,3095\n1,3,8352,2820,1293,779,656,144\n1,3,16705,2037,3202,10643,116,1365\n1,3,18291,1266,21042,5373,4173,14472\n1,3,4420,5139,2661,8872,1321,181\n2,3,19899,5332,8713,8132,764,648\n2,3,8190,6343,9794,1285,1901,1780\n1,3,20398,1137,3,4407,3,975\n1,3,717,3587,6532,7530,529,894\n2,3,12205,12697,28540,869,12034,1009\n1,3,10766,1175,2067,2096,301,167\n1,3,1640,3259,3655,868,1202,1653\n1,3,7005,829,3009,430,610,529\n2,3,219,9540,14403,283,7818,156\n2,3,10362,9232,11009,737,3537,2342\n1,3,20874,1563,1783,2320,550,772\n2,3,11867,3327,4814,1178,3837,120\n2,3,16117,46197,92780,1026,40827,2944\n2,3,22925,73498,32114,987,20070,903\n1,3,43265,5025,8117,6312,1579,14351\n1,3,7864,542,4042,9735,165,46\n1,3,24904,3836,5330,3443,454,3178\n1,3,11405,596,1638,3347,69,360\n1,3,12754,2762,2530,8693,627,1117\n2,3,9198,27472,32034,3232,18906,5130\n1,3,11314,3090,2062,35009,71,2698\n2,3,5626,12220,11323,206,5038,244\n1,3,3,2920,6252,440,223,709\n2,3,23,2616,8118,145,3874,217\n1,3,403,254,610,774,54,63\n1,3,503,112,778,895,56,132\n1,3,9658,2182,1909,5639,215,323\n2,3,11594,7779,12144,3252,8035,3029\n2,3,1420,10810,16267,1593,6766,1838\n2,3,2932,6459,7677,2561,4573,1386\n1,3,56082,3504,8906,18028,1480,2498\n1,3,14100,2132,3445,1336,1491,548\n1,3,15587,1014,3970,910,139,1378\n2,3,1454,6337,10704,133,6830,1831\n2,3,8797,10646,14886,2471,8969,1438\n2,3,1531,8397,6981,247,2505,1236\n2,3,1406,16729,28986,673,836,3\n1,3,11818,1648,1694,2276,169,1647\n2,3,12579,11114,17569,805,6457,1519\n1,3,19046,2770,2469,8853,483,2708\n1,3,14438,2295,1733,3220,585,1561\n1,3,18044,1080,2000,2555,118,1266\n1,3,11134,793,2988,2715,276,610\n1,3,11173,2521,3355,1517,310,222\n1,3,6990,3880,5380,1647,319,1160\n1,3,20049,1891,2362,5343,411,933\n1,3,8258,2344,2147,3896,266,635\n1,3,17160,1200,3412,2417,174,1136\n1,3,4020,3234,1498,2395,264,255\n1,3,12212,201,245,1991,25,860\n2,3,11170,10769,8814,2194,1976,143\n1,3,36050,1642,2961,4787,500,1621\n1,3,76237,3473,7102,16538,778,918\n1,3,19219,1840,1658,8195,349,483\n2,3,21465,7243,10685,880,2386,2749\n1,3,140,8847,3823,142,1062,3\n1,3,42312,926,1510,1718,410,1819\n1,3,7149,2428,699,6316,395,911\n1,3,2101,589,314,346,70,310\n1,3,14903,2032,2479,576,955,328\n1,3,9434,1042,1235,436,256,396\n1,3,7388,1882,2174,720,47,537\n1,3,6300,1289,2591,1170,199,326\n1,3,4625,8579,7030,4575,2447,1542\n1,3,3087,8080,8282,661,721,36\n1,3,13537,4257,5034,155,249,3271\n1,3,5387,4979,3343,825,637,929\n1,3,17623,4280,7305,2279,960,2616\n1,3,30379,13252,5189,321,51,1450\n1,3,37036,7152,8253,2995,20,3\n1,3,10405,1596,1096,8425,399,318\n1,3,18827,3677,1988,118,516,201\n2,3,22039,8384,34792,42,12591,4430\n1,3,7769,1936,2177,926,73,520\n1,3,9203,3373,2707,1286,1082,526\n1,3,5924,584,542,4052,283,434\n1,3,31812,1433,1651,800,113,1440\n1,3,16225,1825,1765,853,170,1067\n1,3,1289,3328,2022,531,255,1774\n1,3,18840,1371,3135,3001,352,184\n1,3,3463,9250,2368,779,302,1627\n1,3,622,55,137,75,7,8\n2,3,1989,10690,19460,233,11577,2153\n2,3,3830,5291,14855,317,6694,3182\n1,3,17773,1366,2474,3378,811,418\n2,3,2861,6570,9618,930,4004,1682\n2,3,355,7704,14682,398,8077,303\n2,3,1725,3651,12822,824,4424,2157\n1,3,12434,540,283,1092,3,2233\n1,3,15177,2024,3810,2665,232,610\n2,3,5531,15726,26870,2367,13726,446\n2,3,5224,7603,8584,2540,3674,238\n2,3,15615,12653,19858,4425,7108,2379\n2,3,4822,6721,9170,993,4973,3637\n1,3,2926,3195,3268,405,1680,693\n1,3,5809,735,803,1393,79,429\n1,3,5414,717,2155,2399,69,750\n2,3,260,8675,13430,1116,7015,323\n2,3,200,25862,19816,651,8773,6250\n1,3,955,5479,6536,333,2840,707\n2,3,514,7677,19805,937,9836,716\n1,3,286,1208,5241,2515,153,1442\n2,3,2343,7845,11874,52,4196,1697\n1,3,45640,6958,6536,7368,1532,230\n1,3,12759,7330,4533,1752,20,2631\n1,3,11002,7075,4945,1152,120,395\n1,3,3157,4888,2500,4477,273,2165\n1,3,12356,6036,8887,402,1382,2794\n1,3,112151,29627,18148,16745,4948,8550\n1,3,694,8533,10518,443,6907,156\n1,3,36847,43950,20170,36534,239,47943\n1,3,327,918,4710,74,334,11\n1,3,8170,6448,1139,2181,58,247\n1,3,3009,521,854,3470,949,727\n1,3,2438,8002,9819,6269,3459,3\n2,3,8040,7639,11687,2758,6839,404\n2,3,834,11577,11522,275,4027,1856\n1,3,16936,6250,1981,7332,118,64\n1,3,13624,295,1381,890,43,84\n1,3,5509,1461,2251,547,187,409\n2,3,180,3485,20292,959,5618,666\n1,3,7107,1012,2974,806,355,1142\n1,3,17023,5139,5230,7888,330,1755\n1,1,30624,7209,4897,18711,763,2876\n2,1,2427,7097,10391,1127,4314,1468\n1,1,11686,2154,6824,3527,592,697\n1,1,9670,2280,2112,520,402,347\n2,1,3067,13240,23127,3941,9959,731\n2,1,4484,14399,24708,3549,14235,1681\n1,1,25203,11487,9490,5065,284,6854\n1,1,583,685,2216,469,954,18\n1,1,1956,891,5226,1383,5,1328\n2,1,1107,11711,23596,955,9265,710\n1,1,6373,780,950,878,288,285\n2,1,2541,4737,6089,2946,5316,120\n1,1,1537,3748,5838,1859,3381,806\n2,1,5550,12729,16767,864,12420,797\n1,1,18567,1895,1393,1801,244,2100\n2,1,12119,28326,39694,4736,19410,2870\n1,1,7291,1012,2062,1291,240,1775\n1,1,3317,6602,6861,1329,3961,1215\n2,1,2362,6551,11364,913,5957,791\n1,1,2806,10765,15538,1374,5828,2388\n2,1,2532,16599,36486,179,13308,674\n1,1,18044,1475,2046,2532,130,1158\n2,1,18,7504,15205,1285,4797,6372\n1,1,4155,367,1390,2306,86,130\n1,1,14755,899,1382,1765,56,749\n1,1,5396,7503,10646,91,4167,239\n1,1,5041,1115,2856,7496,256,375\n2,1,2790,2527,5265,5612,788,1360\n1,1,7274,659,1499,784,70,659\n1,1,12680,3243,4157,660,761,786\n2,1,20782,5921,9212,1759,2568,1553\n1,1,4042,2204,1563,2286,263,689\n1,1,1869,577,572,950,4762,203\n1,1,8656,2746,2501,6845,694,980\n2,1,11072,5989,5615,8321,955,2137\n1,1,2344,10678,3828,1439,1566,490\n1,1,25962,1780,3838,638,284,834\n1,1,964,4984,3316,937,409,7\n1,1,15603,2703,3833,4260,325,2563\n1,1,1838,6380,2824,1218,1216,295\n1,1,8635,820,3047,2312,415,225\n1,1,18692,3838,593,4634,28,1215\n1,1,7363,475,585,1112,72,216\n1,1,47493,2567,3779,5243,828,2253\n1,1,22096,3575,7041,11422,343,2564\n1,1,24929,1801,2475,2216,412,1047\n1,1,18226,659,2914,3752,586,578\n1,1,11210,3576,5119,561,1682,2398\n1,1,6202,7775,10817,1183,3143,1970\n2,1,3062,6154,13916,230,8933,2784\n1,1,8885,2428,1777,1777,430,610\n1,1,13569,346,489,2077,44,659\n1,1,15671,5279,2406,559,562,572\n1,1,8040,3795,2070,6340,918,291\n1,1,3191,1993,1799,1730,234,710\n2,1,6134,23133,33586,6746,18594,5121\n1,1,6623,1860,4740,7683,205,1693\n1,1,29526,7961,16966,432,363,1391\n1,1,10379,17972,4748,4686,1547,3265\n1,1,31614,489,1495,3242,111,615\n1,1,11092,5008,5249,453,392,373\n1,1,8475,1931,1883,5004,3593,987\n1,1,56083,4563,2124,6422,730,3321\n1,1,53205,4959,7336,3012,967,818\n1,1,9193,4885,2157,327,780,548\n1,1,7858,1110,1094,6818,49,287\n1,1,23257,1372,1677,982,429,655\n1,1,2153,1115,6684,4324,2894,411\n2,1,1073,9679,15445,61,5980,1265\n1,1,5909,23527,13699,10155,830,3636\n2,1,572,9763,22182,2221,4882,2563\n1,1,20893,1222,2576,3975,737,3628\n2,1,11908,8053,19847,1069,6374,698\n1,1,15218,258,1138,2516,333,204\n1,1,4720,1032,975,5500,197,56\n1,1,2083,5007,1563,1120,147,1550\n1,1,514,8323,6869,529,93,1040\n1,3,36817,3045,1493,4802,210,1824\n1,3,894,1703,1841,744,759,1153\n1,3,680,1610,223,862,96,379\n1,3,27901,3749,6964,4479,603,2503\n1,3,9061,829,683,16919,621,139\n1,3,11693,2317,2543,5845,274,1409\n2,3,17360,6200,9694,1293,3620,1721\n1,3,3366,2884,2431,977,167,1104\n2,3,12238,7108,6235,1093,2328,2079\n1,3,49063,3965,4252,5970,1041,1404\n1,3,25767,3613,2013,10303,314,1384\n1,3,68951,4411,12609,8692,751,2406\n1,3,40254,640,3600,1042,436,18\n1,3,7149,2247,1242,1619,1226,128\n1,3,15354,2102,2828,8366,386,1027\n1,3,16260,594,1296,848,445,258\n1,3,42786,286,471,1388,32,22\n1,3,2708,2160,2642,502,965,1522\n1,3,6022,3354,3261,2507,212,686\n1,3,2838,3086,4329,3838,825,1060\n2,2,3996,11103,12469,902,5952,741\n1,2,21273,2013,6550,909,811,1854\n2,2,7588,1897,5234,417,2208,254\n1,2,19087,1304,3643,3045,710,898\n2,2,8090,3199,6986,1455,3712,531\n2,2,6758,4560,9965,934,4538,1037\n1,2,444,879,2060,264,290,259\n2,2,16448,6243,6360,824,2662,2005\n2,2,5283,13316,20399,1809,8752,172\n2,2,2886,5302,9785,364,6236,555\n2,2,2599,3688,13829,492,10069,59\n2,2,161,7460,24773,617,11783,2410\n2,2,243,12939,8852,799,3909,211\n2,2,6468,12867,21570,1840,7558,1543\n1,2,17327,2374,2842,1149,351,925\n1,2,6987,1020,3007,416,257,656\n2,2,918,20655,13567,1465,6846,806\n1,2,7034,1492,2405,12569,299,1117\n1,2,29635,2335,8280,3046,371,117\n2,2,2137,3737,19172,1274,17120,142\n1,2,9784,925,2405,4447,183,297\n1,2,10617,1795,7647,1483,857,1233\n2,2,1479,14982,11924,662,3891,3508\n1,2,7127,1375,2201,2679,83,1059\n1,2,1182,3088,6114,978,821,1637\n1,2,11800,2713,3558,2121,706,51\n2,2,9759,25071,17645,1128,12408,1625\n1,2,1774,3696,2280,514,275,834\n1,2,9155,1897,5167,2714,228,1113\n1,2,15881,713,3315,3703,1470,229\n1,2,13360,944,11593,915,1679,573\n1,2,25977,3587,2464,2369,140,1092\n1,2,32717,16784,13626,60869,1272,5609\n1,2,4414,1610,1431,3498,387,834\n1,2,542,899,1664,414,88,522\n1,2,16933,2209,3389,7849,210,1534\n1,2,5113,1486,4583,5127,492,739\n1,2,9790,1786,5109,3570,182,1043\n2,2,11223,14881,26839,1234,9606,1102\n1,2,22321,3216,1447,2208,178,2602\n2,2,8565,4980,67298,131,38102,1215\n2,2,16823,928,2743,11559,332,3486\n2,2,27082,6817,10790,1365,4111,2139\n1,2,13970,1511,1330,650,146,778\n1,2,9351,1347,2611,8170,442,868\n1,2,3,333,7021,15601,15,550\n1,2,2617,1188,5332,9584,573,1942\n2,3,381,4025,9670,388,7271,1371\n2,3,2320,5763,11238,767,5162,2158\n1,3,255,5758,5923,349,4595,1328\n2,3,1689,6964,26316,1456,15469,37\n1,3,3043,1172,1763,2234,217,379\n1,3,1198,2602,8335,402,3843,303\n2,3,2771,6939,15541,2693,6600,1115\n2,3,27380,7184,12311,2809,4621,1022\n1,3,3428,2380,2028,1341,1184,665\n2,3,5981,14641,20521,2005,12218,445\n1,3,3521,1099,1997,1796,173,995\n2,3,1210,10044,22294,1741,12638,3137\n1,3,608,1106,1533,830,90,195\n2,3,117,6264,21203,228,8682,1111\n1,3,14039,7393,2548,6386,1333,2341\n1,3,190,727,2012,245,184,127\n1,3,22686,134,218,3157,9,548\n2,3,37,1275,22272,137,6747,110\n1,3,759,18664,1660,6114,536,4100\n1,3,796,5878,2109,340,232,776\n1,3,19746,2872,2006,2601,468,503\n1,3,4734,607,864,1206,159,405\n1,3,2121,1601,2453,560,179,712\n1,3,4627,997,4438,191,1335,314\n1,3,2615,873,1524,1103,514,468\n2,3,4692,6128,8025,1619,4515,3105\n1,3,9561,2217,1664,1173,222,447\n1,3,3477,894,534,1457,252,342\n1,3,22335,1196,2406,2046,101,558\n1,3,6211,337,683,1089,41,296\n2,3,39679,3944,4955,1364,523,2235\n1,3,20105,1887,1939,8164,716,790\n1,3,3884,3801,1641,876,397,4829\n2,3,15076,6257,7398,1504,1916,3113\n1,3,6338,2256,1668,1492,311,686\n1,3,5841,1450,1162,597,476,70\n2,3,3136,8630,13586,5641,4666,1426\n1,3,38793,3154,2648,1034,96,1242\n1,3,3225,3294,1902,282,68,1114\n2,3,4048,5164,10391,130,813,179\n1,3,28257,944,2146,3881,600,270\n1,3,17770,4591,1617,9927,246,532\n1,3,34454,7435,8469,2540,1711,2893\n1,3,1821,1364,3450,4006,397,361\n1,3,10683,21858,15400,3635,282,5120\n1,3,11635,922,1614,2583,192,1068\n1,3,1206,3620,2857,1945,353,967\n1,3,20918,1916,1573,1960,231,961\n1,3,9785,848,1172,1677,200,406\n1,3,9385,1530,1422,3019,227,684\n1,3,3352,1181,1328,5502,311,1000\n1,3,2647,2761,2313,907,95,1827\n1,3,518,4180,3600,659,122,654\n1,3,23632,6730,3842,8620,385,819\n1,3,12377,865,3204,1398,149,452\n1,3,9602,1316,1263,2921,841,290\n2,3,4515,11991,9345,2644,3378,2213\n1,3,11535,1666,1428,6838,64,743\n1,3,11442,1032,582,5390,74,247\n1,3,9612,577,935,1601,469,375\n1,3,4446,906,1238,3576,153,1014\n1,3,27167,2801,2128,13223,92,1902\n1,3,26539,4753,5091,220,10,340\n1,3,25606,11006,4604,127,632,288\n1,3,18073,4613,3444,4324,914,715\n1,3,6884,1046,1167,2069,593,378\n1,3,25066,5010,5026,9806,1092,960\n2,3,7362,12844,18683,2854,7883,553\n2,3,8257,3880,6407,1646,2730,344\n1,3,8708,3634,6100,2349,2123,5137\n1,3,6633,2096,4563,1389,1860,1892\n1,3,2126,3289,3281,1535,235,4365\n1,3,97,3605,12400,98,2970,62\n1,3,4983,4859,6633,17866,912,2435\n1,3,5969,1990,3417,5679,1135,290\n2,3,7842,6046,8552,1691,3540,1874\n2,3,4389,10940,10908,848,6728,993\n1,3,5065,5499,11055,364,3485,1063\n2,3,660,8494,18622,133,6740,776\n1,3,8861,3783,2223,633,1580,1521\n1,3,4456,5266,13227,25,6818,1393\n2,3,17063,4847,9053,1031,3415,1784\n1,3,26400,1377,4172,830,948,1218\n2,3,17565,3686,4657,1059,1803,668\n2,3,16980,2884,12232,874,3213,249\n1,3,11243,2408,2593,15348,108,1886\n1,3,13134,9347,14316,3141,5079,1894\n1,3,31012,16687,5429,15082,439,1163\n1,3,3047,5970,4910,2198,850,317\n1,3,8607,1750,3580,47,84,2501\n1,3,3097,4230,16483,575,241,2080\n1,3,8533,5506,5160,13486,1377,1498\n1,3,21117,1162,4754,269,1328,395\n1,3,1982,3218,1493,1541,356,1449\n1,3,16731,3922,7994,688,2371,838\n1,3,29703,12051,16027,13135,182,2204\n1,3,39228,1431,764,4510,93,2346\n2,3,14531,15488,30243,437,14841,1867\n1,3,10290,1981,2232,1038,168,2125\n1,3,2787,1698,2510,65,477,52\n'

URL = 'https://archive.ics.uci.edu/static/public/292/wholesale+customers.zip'
try:
    req = urllib.request.Request(URL, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=20) as response:
        archive_bytes = response.read()
    with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
        csv_name = next(name for name in archive.namelist() if name.endswith('.csv'))
        raw_bytes = archive.read(csv_name)
    source = 'UCI (tải trực tiếp)'
except Exception as error:
    raw_bytes = EMBEDDED_CSV.encode('utf-8')
    source = f'Bản nhúng sẵn trong notebook ({type(error).__name__})'

(PROJECT_DIR / 'data' / 'wholesale_customers.csv').write_bytes(raw_bytes)
print('Nguồn dữ liệu:', source)

In [ ]:
# Ghi mã nguồn ứng dụng dùng chung (pipeline.py, app.py, storage.py, experience.py, retrain.py)
PROJECT_FILES = {'pipeline.py': '"""Pipeline phân cụm dùng chung cho notebook Colab, giao diện Streamlit và lịch huấn luyện lại.\n\nĐề tài: Phân khúc khách hàng siêu thị để gợi ý khuyến mãi (Bài toán Phân cụm).\nDữ liệu: Wholesale Customers (UCI, 440 dòng, 6 nhóm chi tiêu + Channel, Region).\nLuồng chuẩn: validate -> loại ngoại lai (IQR) -> chuẩn hóa (StandardScaler / MinMaxScaler)\n             -> K-Means / Hierarchical (Agglomerative) / DBSCAN -> đánh giá (Elbow, Silhouette)\n             -> hồ sơ phân khúc & gợi ý khuyến mãi -> lưu phiên kết quả.\n"""\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nimport time\nimport uuid\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN\nfrom sklearn.preprocessing import StandardScaler, MinMaxScaler, FunctionTransformer\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score\nfrom sklearn.decomposition import PCA\n\nFEATURES = [\'Fresh\', \'Milk\', \'Grocery\', \'Frozen\', \'Detergents_Paper\', \'Delicassen\']\nCOLUMNS = [\'Channel\', \'Region\'] + FEATURES\nCATEGORY_VI = {\n    \'Fresh\': \'Hàng tươi sống\',\n    \'Milk\': \'Sữa & chế phẩm\',\n    \'Grocery\': \'Tạp hóa\',\n    \'Frozen\': \'Hàng đông lạnh\',\n    \'Detergents_Paper\': \'Chất tẩy rửa & giấy\',\n    \'Delicassen\': \'Thực phẩm chế biến\',\n}\nFOOD = [\'Fresh\', \'Frozen\', \'Delicassen\']\nRETAIL = [\'Milk\', \'Grocery\', \'Detergents_Paper\']\nROOT = Path(__file__).resolve().parent\nSOURCE = ROOT / \'data\' / \'wholesale_customers.csv\'\n\n# Bắt buộc >= 3 cụm để phân khúc có ý nghĩa thiết kế khuyến mãi (k=2 chỉ tách 2 kênh Channel có sẵn).\nMIN_SEGMENTS = 3\n\n\n# --------------------------------------------------------------------------- 1. Kiểm tra dữ liệu\ndef validate(raw):\n    if raw.columns.duplicated().any():\n        raise ValueError(\'Tên cột bị lặp; cần mỗi cột xuất hiện đúng một lần.\')\n    missing = set(COLUMNS) - set(raw.columns)\n    if missing:\n        raise ValueError(f\'Thiếu cột: {sorted(missing)}\')\n    frame = raw[COLUMNS].apply(pd.to_numeric, errors=\'coerce\')\n    if frame.empty:\n        raise ValueError(\'Dữ liệu rỗng.\')\n    if not np.isfinite(frame.to_numpy(dtype=float)).all():\n        raise ValueError(\'Dữ liệu có ô thiếu, chữ hoặc giá trị vô hạn. Hãy sửa trước khi lưu.\')\n    if (frame[FEATURES] < 0).any().any():\n        raise ValueError(\'Chi tiêu không được âm.\')\n    if (frame[FEATURES].sum(axis=1) == 0).any():\n        raise ValueError(\'Mỗi khách hàng cần có chi tiêu dương ở ít nhất một nhóm hàng.\')\n    if not frame.Channel.isin([1, 2]).all() or not frame.Region.isin([1, 2, 3]).all():\n        raise ValueError(\'Channel chỉ nhận 1/2; Region chỉ nhận 1/2/3.\')\n    frame[[\'Channel\', \'Region\']] = frame[[\'Channel\', \'Region\']].astype(int)\n    frame[FEATURES] = frame[FEATURES].astype(float)\n    return frame.reset_index(drop=True)\n\n\ndef data_fingerprint(frame):\n    """Mã băm chuẩn hóa dùng chung giữa CSV, SQLite, ứng dụng và notebook."""\n    clean = validate(frame)\n    payload = clean.to_json(orient=\'split\', index=False, double_precision=15)\n    return hashlib.sha256(payload.encode(\'utf-8\')).hexdigest()\n\n\n# --------------------------------------------------------------------------- 2. Tiền xử lý (IQR + Scaling)\ndef detect_outliers_iqr(frame, method=\'iqr\'):\n    """Phát hiện ngoại lai theo quy tắc IQR (1.5 * IQR).\n    - \'iqr\': tính IQR trực tiếp trên dữ liệu chi tiêu gốc.\n    - \'iqr_log\': tính IQR trên thang log1p (giữ lại nhiều điểm hơn).\n    - \'none\': không loại ngoại lai.\n    """\n    if method == \'none\':\n        return np.zeros(len(frame), dtype=bool), None\n    values = np.log1p(frame[FEATURES]) if method == \'iqr_log\' else frame[FEATURES].astype(float)\n    q1 = values.quantile(0.25)\n    q3 = values.quantile(0.75)\n    iqr = q3 - q1\n    lower = q1 - 1.5 * iqr\n    upper = q3 + 1.5 * iqr\n    mask = ((values < lower) | (values > upper)).any(axis=1).to_numpy()\n    if (~mask).sum() < 10:\n        return np.zeros(len(frame), dtype=bool), None\n    bounds = {\'method\': method, \'lower\': lower.to_dict(), \'upper\': upper.to_dict()}\n    return mask, bounds\n\n\ndef make_scaler(scaler_type=\'standard\', use_log=False):\n    """Tạo bộ chuẩn hóa dữ liệu: StandardScaler hoặc MinMaxScaler (có thể kèm log1p)."""\n    scaler = MinMaxScaler() if scaler_type == \'minmax\' else StandardScaler()\n    if use_log:\n        return Pipeline([(\'log\', FunctionTransformer(np.log1p, validate=True)), (\'scale\', scaler)])\n    return Pipeline([(\'scale\', scaler)])\n\n\ndef prepare(frame, scaler_type=\'standard\', outlier_method=\'iqr\', use_log=False):\n    """Tiền xử lý: phát hiện ngoại lai bằng IQR trên tập sạch -> fit scaler trên dữ liệu không ngoại lai."""\n    outlier, bounds = detect_outliers_iqr(frame, method=outlier_method)\n    prep = make_scaler(scaler_type=scaler_type, use_log=use_log)\n    prep.fit(frame.loc[~outlier, FEATURES])\n    X = prep.transform(frame[FEATURES])\n    return prep, X, outlier, bounds\n\n\n# --------------------------------------------------------------------------- 3. Đánh giá & Thuật toán\ndef evaluate(X, labels):\n    mask = labels != -1\n    count = len(set(labels[mask]))\n    coverage = float(mask.mean())\n    result = dict(clusters=count, noise_rate=1 - coverage, silhouette=np.nan, davies_bouldin=np.nan, calinski_harabasz=np.nan)\n    if 2 <= count < int(mask.sum()):\n        result.update(\n            silhouette=float(silhouette_score(X[mask], labels[mask])),\n            davies_bouldin=float(davies_bouldin_score(X[mask], labels[mask])),\n            calinski_harabasz=float(calinski_harabasz_score(X[mask], labels[mask])),\n        )\n    return result\n\n\ndef elbow_k(ks, inertias):\n    """Điểm khuỷu (Elbow Method): điểm xa nhất so với đường thẳng nối hai đầu đường cong inertia."""\n    x = np.asarray(ks, dtype=float)\n    y = np.asarray(inertias, dtype=float)\n    if len(x) < 3 or y[0] == y[-1]:\n        return int(x[0])\n    xn = (x - x[0]) / (x[-1] - x[0])\n    yn = (y - y[-1]) / (y[0] - y[-1])\n    return int(x[int(np.argmax((1 - xn) - yn))])\n\n\ndef make_model(algorithm, params):\n    if algorithm == \'KMeans\':\n        return KMeans(n_clusters=int(params[\'k\']), random_state=42, n_init=20)\n    if algorithm == \'Hierarchical\':\n        return AgglomerativeClustering(n_clusters=int(params[\'k\']), linkage=\'ward\')\n    if algorithm == \'DBSCAN\':\n        return DBSCAN(eps=float(params[\'eps\']), min_samples=int(params[\'min_samples\']))\n    raise ValueError(\'Thuật toán không hợp lệ\')\n\n\ndef dbscan_grid(scaler_type):\n    """Lưới tham số DBSCAN phù hợp với thang đo của StandardScaler và MinMaxScaler."""\n    if scaler_type == \'minmax\':\n        eps_values = [0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5]\n    else:\n        eps_values = [0.75, 1.0, 1.1, 1.2, 1.35, 1.5, 1.75]\n    return [(\'DBSCAN\', {\'eps\': eps, \'min_samples\': m}) for eps in eps_values for m in [3, 4, 5, 6]]\n\n\n@dataclass\nclass SearchResult:\n    preprocessor: object\n    X: np.ndarray\n    outlier: np.ndarray\n    bounds: dict\n    table: pd.DataFrame\n    candidates: dict = field(default_factory=dict)\n    best: pd.DataFrame = None\n    selected: int = 0\n    elbow: int = 0\n    scaler_type: str = \'standard\'\n    outlier_method: str = \'iqr\'\n    use_log: bool = False\n\n    @property\n    def X_fit(self):\n        return self.X[~self.outlier]\n\n\ndef search(frame, scaler_type=\'standard\', outlier_method=\'iqr\', use_log=False):\n    frame = validate(frame)\n    if len(frame) < 15:\n        raise ValueError(\'Cần ít nhất 15 khách hàng để khảo sát phân cụm.\')\n    prep, X, outlier, bounds = prepare(frame, scaler_type=scaler_type, outlier_method=outlier_method, use_log=use_log)\n    X_fit = X[~outlier]\n    rows, candidates = [], {}\n    kmax = min(9, len(X_fit))\n    configs = [(name, {\'k\': k}) for name in [\'KMeans\', \'Hierarchical\'] for k in range(2, kmax)]\n    configs += dbscan_grid(scaler_type)\n    for i, (algorithm, params) in enumerate(configs):\n        model = make_model(algorithm, params)\n        start = time.perf_counter()\n        labels = model.fit_predict(X_fit)\n        elapsed = time.perf_counter() - start\n        scores = evaluate(X_fit, labels)\n        counts = pd.Series(labels[labels != -1]).value_counts()\n        eligible = (\n            scores[\'clusters\'] >= MIN_SEGMENTS\n            and scores[\'noise_rate\'] <= 0.30\n            and len(counts) > 0\n            and counts.min() >= 5\n            and np.isfinite(scores[\'silhouette\'])\n        )\n        rows.append(dict(\n            candidate=i, algorithm=algorithm, params=json.dumps(params), seconds=elapsed,\n            inertia=float(model.inertia_) if algorithm == \'KMeans\' else np.nan,\n            min_cluster=int(counts.min()) if len(counts) else 0, eligible=bool(eligible), **scores,\n        ))\n        candidates[i] = (model, labels)\n    table = pd.DataFrame(rows)\n    ranked = table[table.eligible].sort_values([\'silhouette\', \'davies_bouldin\'], ascending=[False, True])\n    if ranked.empty:\n        # Dự phòng nếu tập dữ liệu thử nghiệm nhỏ không đạt ngưỡng 3 cụm\n        ranked = table[table.silhouette.notna()].sort_values([\'silhouette\', \'davies_bouldin\'], ascending=[False, True])\n    if ranked.empty:\n        raise ValueError(\'Không có cấu hình phân cụm hợp lệ. Hãy kiểm tra dữ liệu.\')\n    representatives = []\n    for name in [\'KMeans\', \'Hierarchical\', \'DBSCAN\']:\n        options = ranked[ranked.algorithm == name]\n        if options.empty:\n            options = table[table.algorithm == name].sort_values([\'silhouette\', \'davies_bouldin\'], ascending=[False, True], na_position=\'last\')\n        representatives.append(options.head(1))\n    best = pd.concat(representatives, ignore_index=True)\n    # Ưu tiên chọn phương pháp phân cụm toàn bộ (KMeans/Hierarchical) làm mô hình chính nếu ở chế độ Auto\n    full_cov = ranked[ranked.algorithm.isin([\'KMeans\', \'Hierarchical\'])]\n    selected_id = int(full_cov.iloc[0].candidate) if not full_cov.empty else int(ranked.iloc[0].candidate)\n    km = table[table.algorithm == \'KMeans\']\n    elbow = elbow_k(km.params.map(lambda p: json.loads(p)[\'k\']), km.inertia)\n    return SearchResult(prep, X, outlier, bounds, table, candidates, best, selected_id, elbow, scaler_type, outlier_method, use_log)\n\n\ndef centroids(X, labels):\n    return {int(c): X[labels == c].mean(axis=0) for c in sorted(set(labels)) if c != -1}\n\n\ndef assign_nearest(X, cents):\n    keys = sorted(cents)\n    matrix = np.array([cents[k] for k in keys])\n    dist = np.linalg.norm(X[:, None, :] - matrix[None, :, :], axis=2)\n    return np.array(keys)[dist.argmin(axis=1)], dist.min(axis=1)\n\n\ndef full_labels(res, labels_fit):\n    """Nhãn cho mọi khách hàng. Khách ngoại lai (bị loại lúc train) được gán vào cụm gần nhất để tham khảo."""\n    labels = np.full(len(res.X), -1, dtype=int)\n    labels[~res.outlier] = labels_fit\n    cents = centroids(res.X_fit, labels_fit)\n    if res.outlier.any() and cents:\n        labels[res.outlier] = assign_nearest(res.X[res.outlier], cents)[0]\n    return labels\n\n\ndef compare_scalers(frame, outlier_method=\'iqr\'):\n    """So sánh 2 bộ chuẩn hóa theo đúng đề cương: StandardScaler vs MinMaxScaler."""\n    parts = []\n    for scaler_type, label in [(\'standard\', \'StandardScaler (Z-score)\'), (\'minmax\', \'MinMaxScaler [0, 1]\')]:\n        res = search(frame, scaler_type=scaler_type, outlier_method=outlier_method, use_log=False)\n        part = res.best.copy()\n        part.insert(0, \'scaler\', label)\n        part[\'rows_used\'] = int((~res.outlier).sum())\n        part[\'elbow_k\'] = res.elbow\n        parts.append(part)\n    return pd.concat(parts, ignore_index=True)\n\n\n# --------------------------------------------------------------------------- 4. Hồ sơ phân khúc & Gợi ý khuyến mãi\ndef describe_segment(cluster, ratio_row, total_ratio, retail_share):\n    """Đặt tên phân khúc và gợi ý khuyến mãi siêu thị dựa trên mức chi tiêu tương đối."""\n    top = ratio_row.dropna().sort_values(ascending=False).head(2).index\n    top_vi = \', \'.join(CATEGORY_VI[t] for t in top)\n    food, retail = float(ratio_row[FOOD].mean()), float(ratio_row[RETAIL].mean())\n    if total_ratio < 0.78:\n        kind = \'Khách mua sắm tiết kiệm / quy mô nhỏ\'\n        offer = f\'Tặng Voucher giảm giá cho đơn hàng tiếp theo và Combo tiết kiệm nhóm {top_vi} để tăng tần suất mua.\'\n    elif total_ratio > 1.35 and abs(food - retail) < 0.45 * max(food, retail):\n        kind = \'Khách hàng lớn đa ngành (VIP)\'\n        offer = \'Áp dụng thẻ thành viên VIP, chiết khấu bậc thang theo hóa đơn lớn và miễn phí giao hàng tận nơi.\'\n    elif retail > 1.25 * food:\n        kind = \'Nhóm ưu tiên Tạp hóa & Gia dụng\'\n        offer = f\'Khuyến mãi mua nhiều giảm giá (mua 2 tặng 1 / giảm 10% theo thùng) cho nhóm {top_vi}.\'\n    elif food > 1.25 * retail:\n        kind = \'Nhóm ưu tiên Thực phẩm Tươi & Đông lạnh\'\n        offer = f\'Ưu đãi Combo thực phẩm tươi sống + đông lạnh hàng tuần, giảm giá giờ vàng cho {top_vi}.\'\n    else:\n        kind = \'Khách hàng cân bằng nhiều nhóm hàng\'\n        offer = f\'Ưu đãi theo ngưỡng giá trị hóa đơn kèm quà tặng dùng thử ở nhóm {top_vi}.\'\n    channel = f\'{retail_share:.0%} thuộc kênh bán lẻ\' if retail_share == retail_share else \'\'\n    rationale = f\'Chi tiêu trung bình bằng {total_ratio:.2f} lần mức chung; nổi bật nhất: {top_vi}.\' + (f\' {channel}.\' if channel else \'\')\n    return f\'Nhóm {cluster} — {kind}\', offer, rationale\n\n\nNOISE_TEXT = \'Điểm nhiễu (DBSCAN): xem xét riêng trước khi gửi khuyến mãi.\'\nOUTLIER_TEXT = \'Khách hàng chi tiêu đột biến (Ngoại lai IQR): chăm sóc riêng / ưu đãi VIP theo hợp đồng. (Gần với {segment}).\'\n\n\ndef profiles(frame, labels, outlier=None):\n    outlier = np.zeros(len(frame), dtype=bool) if outlier is None else np.asarray(outlier, dtype=bool)\n    data = frame.copy()\n    data[\'cluster\'] = labels\n    core = data[~outlier]\n    means = core.groupby(\'cluster\')[FEATURES].mean()\n    medians = core.groupby(\'cluster\')[FEATURES].median()\n    overall = core[FEATURES].mean().replace(0, np.nan)\n    ratios = means.div(overall)\n    total_ratio = means.sum(axis=1) / overall.sum()\n    retail_share = core.groupby(\'cluster\').Channel.apply(lambda s: float((s == 2).mean()))\n    rows = []\n    for cluster in means.index:\n        count = int((labels == cluster).sum())\n        out_count = int(((labels == cluster) & outlier).sum())\n        if cluster == -1:\n            segment, offer, rationale = \'Ngoại lệ / nhiễu (DBSCAN)\', NOISE_TEXT, \'Không nằm trong cụm mật độ nào.\'\n        else:\n            segment, offer, rationale = describe_segment(int(cluster), ratios.loc[cluster], float(total_ratio[cluster]), float(retail_share[cluster]))\n        rows.append({\n            \'cluster\': int(cluster), \'customers\': count, \'outliers_assigned\': out_count,\n            \'segment\': segment, \'suggestion\': offer, \'rationale\': rationale,\n        })\n    return pd.DataFrame(rows), means, medians, ratios\n\n\ndef build_assignments(frame, labels, outlier, coords, segments):\n    out = frame.copy()\n    out.insert(0, \'row_id\', np.arange(1, len(frame) + 1))\n    out[\'cluster\'] = labels\n    out[\'is_outlier\'] = np.asarray(outlier, dtype=bool)\n    out[[\'PC1\', \'PC2\']] = coords\n    out = out.merge(segments[[\'cluster\', \'segment\', \'suggestion\']], on=\'cluster\', how=\'left\')\n    mask = out.is_outlier.to_numpy()\n    out.loc[mask, \'suggestion\'] = [OUTLIER_TEXT.format(segment=s) for s in out.loc[mask, \'segment\']]\n    return out\n\n\n# --------------------------------------------------------------------------- 5. Huấn luyện và lưu phiên\ndef train(frame, output_root=None, scaler_type=\'standard\', outlier_method=\'iqr\', use_log=False,\n          algorithm=\'Auto\', allow_ineligible=False, compare=True):\n    frame = validate(frame)\n    res = search(frame, scaler_type=scaler_type, outlier_method=outlier_method, use_log=use_log)\n    table, best, chosen = res.table, res.best, res.selected\n    if algorithm != \'Auto\':\n        options = best[(best.algorithm == algorithm) & (best.eligible | allow_ineligible)]\n        if options.empty:\n            raise ValueError(f\'{algorithm} không có cấu hình đạt tiêu chí.\')\n        chosen = int(options.iloc[0].candidate)\n    selected = table.loc[table.candidate == chosen].iloc[0]\n    model, labels_fit = res.candidates[chosen]\n    labels = full_labels(res, labels_fit)\n    run_id = datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S\') + \'_\' + uuid.uuid4().hex[:8]\n    root = Path(output_root or ROOT / \'artifacts\')\n    folder = root / run_id\n    folder.mkdir(parents=True)\n    segment, means, medians, ratios = profiles(frame, labels, res.outlier)\n    pca = PCA(n_components=2).fit(res.X_fit)\n    coords = pca.transform(res.X)\n    assignments = build_assignments(frame, labels, res.outlier, coords, segment)\n\n    algorithm_root = folder / \'algorithms\'\n    for rep in best.itertuples():\n        target = algorithm_root / rep.algorithm\n        target.mkdir(parents=True)\n        rep_model, rep_fit = res.candidates[int(rep.candidate)]\n        rep_labels = full_labels(res, rep_fit)\n        rep_segments, rep_means, rep_medians, rep_ratios = profiles(frame, rep_labels, res.outlier)\n        rep_assignments = build_assignments(frame, rep_labels, res.outlier, coords, rep_segments)\n        for name, values in {\n            \'assignments\': rep_assignments, \'segments\': rep_segments,\n            \'means\': rep_means.reset_index(), \'medians\': rep_medians.reset_index(),\n            \'relative_spending\': rep_ratios.reset_index(),\n            \'channel_distribution\': pd.crosstab(rep_assignments.cluster, rep_assignments.Channel, normalize=\'index\').reset_index(),\n            \'region_distribution\': pd.crosstab(rep_assignments.cluster, rep_assignments.Region, normalize=\'index\').reset_index(),\n        }.items():\n            values.to_csv(target / f\'{name}.csv\', index=False, encoding=\'utf-8-sig\')\n        joblib.dump({\'preprocessor\': res.preprocessor, \'model\': rep_model, \'pca\': pca,\n                     \'algorithm\': rep.algorithm, \'params\': json.loads(rep.params)}, target / \'model.joblib\')\n\n    extra = {}\n    if compare:\n        extra[\'scaler_comparison\'] = compare_scalers(frame, outlier_method=outlier_method)\n    for name, df in {\n        \'search\': table, \'best_per_algorithm\': best, \'assignments\': assignments, \'segments\': segment,\n        \'means\': means.reset_index(), \'medians\': medians.reset_index(), \'relative_spending\': ratios.reset_index(),\n        **extra,\n    }.items():\n        df.to_csv(folder / f\'{name}.csv\', index=False, encoding=\'utf-8-sig\')\n    frame.to_csv(folder / \'training_snapshot.csv\', index=False, encoding=\'utf-8-sig\')\n\n    digest = data_fingerprint(frame)\n    import sklearn\n    metadata = dict(\n        run_id=run_id, created_utc=datetime.now(timezone.utc).isoformat(), rows=len(frame), data_sha256=digest,\n        algorithm=selected.algorithm, params=json.loads(selected.params),\n        scaler_type=scaler_type, outlier_method=outlier_method, use_log=use_log,\n        eligible=bool(selected.eligible), outliers=int(res.outlier.sum()), elbow_k=res.elbow,\n        selection_policy=f\'Silhouette cao nhất (Davies-Bouldin phá hòa); >={MIN_SEGMENTS} cụm, mỗi cụm >=5, nhiễu <=30%\',\n        metrics=evaluate(res.X_fit, labels_fit), pca_explained_variance=pca.explained_variance_ratio_.tolist(),\n        versions={\'sklearn\': sklearn.__version__, \'numpy\': np.__version__, \'pandas\': pd.__version__},\n    )\n    safe_metadata = json.loads(json.dumps(metadata), parse_constant=lambda _: None)\n    (folder / \'metadata.json\').write_text(json.dumps(safe_metadata, ensure_ascii=False, indent=2, allow_nan=False), encoding=\'utf-8\')\n    joblib.dump({\n        \'preprocessor\': res.preprocessor, \'model\': model, \'pca\': pca, \'metadata\': metadata,\n        \'centroids\': centroids(res.X_fit, labels_fit), \'outlier_bounds\': res.bounds, \'segments\': segment,\n    }, folder / \'model.joblib\')\n\n    report = [\n        \'# Báo cáo phiên huấn luyện\', \'\',\n        f"Phiên: {run_id}; UTC: {metadata[\'created_utc\']}",\n        f"Dữ liệu: {len(frame)} khách hàng (loại {int(res.outlier.sum())} ngoại lai bằng {outlier_method} khi huấn luyện).",\n        f"Chuẩn hóa: {scaler_type}" + (\' + log1p\' if use_log else \'\') + \'.\', \'\',\n        \'## So sánh ba thuật toán\', \'\',\n        \'```text\', best[[\'algorithm\', \'params\', \'clusters\', \'noise_rate\', \'silhouette\', \'davies_bouldin\', \'calinski_harabasz\', \'eligible\']].to_string(index=False), \'```\', \'\',\n        f"Mô hình chính: {selected.algorithm} {selected.params}; Elbow gợi ý k={res.elbow}.", \'\',\n        \'## Hồ sơ phân khúc & gợi ý khuyến mãi\', \'\',\n        \'```text\', segment[[\'cluster\', \'customers\', \'segment\', \'suggestion\']].to_string(index=False), \'```\',\n    ]\n    (folder / \'RUN_REPORT.md\').write_text(\'\\n\'.join(report), encoding=\'utf-8\')\n\n    pointer = root / f\'.latest_{run_id}.tmp\'\n    pointer.write_text(run_id, encoding=\'utf-8\')\n    pointer.replace(root / \'latest.txt\')\n    return folder\n\n\n# --------------------------------------------------------------------------- 6. Dự đoán cho khách mới & Lịch định kỳ\ndef latest_run(root=None):\n    root = Path(root or ROOT / \'artifacts\')\n    pointer = root / \'latest.txt\'\n    if not pointer.exists():\n        return None\n    folder = root / pointer.read_text(encoding=\'utf-8\').strip()\n    return folder if (folder / \'metadata.json\').exists() and (folder / \'model.joblib\').exists() else None\n\n\ndef predict_segments(frame, folder=None):\n    """Gán phân khúc và gợi ý khuyến mãi ngay cho khách hàng mới vừa nhập."""\n    folder = Path(folder) if folder else latest_run()\n    if folder is None or not (folder / \'model.joblib\').exists():\n        return None\n    bundle = joblib.load(folder / \'model.joblib\')\n    if \'centroids\' not in bundle:\n        return None\n    clean = validate(frame)\n    X = bundle[\'preprocessor\'].transform(clean[FEATURES])\n    labels, dist = assign_nearest(X, bundle[\'centroids\'])\n\n    bounds = bundle.get(\'outlier_bounds\')\n    if bounds:\n        vals = np.log1p(clean[FEATURES]) if bounds[\'method\'] == \'iqr_log\' else clean[FEATURES]\n        lower = pd.Series(bounds[\'lower\'])\n        upper = pd.Series(bounds[\'upper\'])\n        is_outlier = ((vals < lower) | (vals > upper)).any(axis=1).to_numpy()\n    else:\n        is_outlier = np.zeros(len(clean), dtype=bool)\n\n    seg = bundle[\'segments\'].set_index(\'cluster\')\n    result = clean.copy()\n    result[\'cluster\'] = labels\n    result[\'segment\'] = seg.loc[labels, \'segment\'].to_numpy()\n    result[\'suggestion\'] = seg.loc[labels, \'suggestion\'].to_numpy()\n    result[\'is_outlier\'] = is_outlier\n    result.loc[is_outlier, \'suggestion\'] = [OUTLIER_TEXT.format(segment=s) for s in result.loc[is_outlier, \'segment\']]\n    result[\'distance_to_centroid\'] = dist\n    return result\n\n\ndef retrain_due(rows_now, root=None, min_new=10, max_days=7, now=None):\n    """Chính sách huấn luyện định kỳ: khi có >= N khách mới hoặc quá X ngày kể từ lần huấn luyện trước."""\n    folder = latest_run(root)\n    if folder is None:\n        return True, \'Chưa có mô hình nào.\'\n    meta = json.loads((folder / \'metadata.json\').read_text(encoding=\'utf-8\'))\n    new = rows_now - int(meta[\'rows\'])\n    age = (now or datetime.now(timezone.utc)) - datetime.fromisoformat(meta[\'created_utc\'])\n    if new >= min_new:\n        return True, f\'Đã có {new} khách hàng mới (ngưỡng {min_new}).\'\n    if new > 0 and age.days >= max_days:\n        return True, f\'Đã {age.days} ngày kể từ lần huấn luyện trước và có {new} khách hàng mới.\'\n    return False, f\'Chưa đến hạn: {max(new, 0)} khách mới (ngưỡng {min_new}), {age.days}/{max_days} ngày.\'\n', 'storage.py': '"""Local SQLite storage. CSV hashes prevent accidental duplicate batch imports."""\nfrom pathlib import Path\nfrom contextlib import contextmanager\nimport os\nimport sqlite3\nimport pandas as pd\nfrom pipeline import ROOT, SOURCE, COLUMNS, validate, data_fingerprint\n\nDB = Path(os.environ.get(\'WHOLESALE_DB\', str(ROOT / \'data\' / \'customers.sqlite3\')))\n\n@contextmanager\ndef connect(db=DB):\n    Path(db).parent.mkdir(parents=True, exist_ok=True)\n    connection = sqlite3.connect(db, timeout=30)\n    try:\n        connection.execute(\'CREATE TABLE IF NOT EXISTS batches (digest TEXT PRIMARY KEY, source TEXT, created TEXT DEFAULT CURRENT_TIMESTAMP)\')\n        columns = \', \'.join(f\'"{c}" REAL NOT NULL\' for c in COLUMNS)\n        connection.execute(f\'CREATE TABLE IF NOT EXISTS customers (id INTEGER PRIMARY KEY, batch TEXT NOT NULL, {columns})\')\n        connection.commit()\n        with connection:\n            yield connection\n    finally:\n        connection.close()\n\ndef add_batch(raw, source, db=DB):\n    frame = validate(raw)\n    digest = data_fingerprint(frame)\n    with connect(db) as conn:\n        # Serialize the duplicate check and insert across local writers.\n        conn.execute(\'BEGIN IMMEDIATE\')\n        if conn.execute(\'SELECT 1 FROM batches WHERE digest=?\', (digest,)).fetchone():\n            return 0\n        conn.execute(\'INSERT INTO batches(digest, source) VALUES (?, ?)\', (digest, source))\n        names = \', \'.join([\'batch\'] + [f\'"{c}"\' for c in COLUMNS])\n        placeholders = \',\'.join(\'?\' for _ in range(len(COLUMNS)+1))\n        conn.executemany(f\'INSERT INTO customers({names}) VALUES ({placeholders})\',\n                         [(digest, *row) for row in frame.itertuples(index=False, name=None)])\n    return len(frame)\n\ndef initialize(db=DB):\n    with connect(db) as conn:\n        # Preserve existing databases created by earlier fingerprint formats.\n        if conn.execute(\'SELECT 1 FROM batches WHERE source=?\', (\'UCI Wholesale Customers 292\',)).fetchone():\n            return 0\n    return add_batch(pd.read_csv(SOURCE), \'UCI Wholesale Customers 292\', db)\n\ndef load_data(db=DB):\n    with connect(db) as conn:\n        frame = pd.read_sql_query(\'SELECT * FROM customers ORDER BY id\', conn)\n    return validate(frame)\n\ndef history(db=DB):\n    with connect(db) as conn:\n        return pd.read_sql_query(\'SELECT source, created, digest FROM batches ORDER BY created DESC\', conn)', 'experience.py': '"""Màn hình hiển thị kết quả phân khúc khách hàng siêu thị và gợi ý khuyến mãi."""\nimport json\nfrom html import escape\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport streamlit as st\n\nfrom pipeline import FEATURES, CATEGORY_VI, latest_run, data_fingerprint\n\nNAMES = CATEGORY_VI\nCOLORS = [\'#2563eb\', \'#0d9488\', \'#9333ea\', \'#ea580c\', \'#db2777\', \'#64748b\', \'#ca8a04\', \'#0891b2\']\n\n\ndef latest_result():\n    folder = latest_run()\n    required = [\'metadata.json\', \'assignments.csv\', \'segments.csv\', \'medians.csv\', \'relative_spending.csv\']\n    return folder if folder is not None and all((folder / name).exists() for name in required) else None\n\n\ndef campaign_draft(customer, profile, ratios, run_id, eligible=True):\n    """Tạo bản nháp gợi ý khuyến mãi theo hồ sơ phân khúc của khách hàng."""\n    cluster = int(customer[\'cluster\'])\n    if cluster == -1 or not eligible:\n        return None\n    top = ratios.loc[cluster, FEATURES].dropna().sort_values(ascending=False).head(2)\n    if top.empty:\n        return None\n    product_names = [NAMES[name] for name in top.index]\n    strategy = profile.get(\'suggestion\') if hasattr(profile, \'get\') else None\n    if not isinstance(strategy, str) or not strategy:\n        strategy = \'Ưu đãi theo ngưỡng đơn hàng cho \' + \' và \'.join(product_names).lower()\n    outlier = bool(customer.get(\'is_outlier\', False)) if hasattr(customer, \'get\') else False\n    if outlier:\n        campaign = \'Chăm sóc VIP / Ngoại lai: Khách hàng có mức chi tiêu đột biến\'\n        message = \'Chuyên viên chăm sóc khách hàng liên hệ trực tiếp để đề xuất chính sách chiết khấu riêng.\'\n    else:\n        campaign = strategy\n        message = (\'Kính mời Quý khách tham khảo chương trình ưu đãi siêu thị dành cho nhóm hàng \'\n                   + \' và \'.join(product_names).lower() + \'. Chi tiết áp dụng tại quầy hoặc đơn đặt hàng tuần này.\')\n    return {\n        \'customer_ref\': f"KH-{int(customer[\'row_id\']):04d}", \'run_id\': run_id, \'cluster\': cluster,\n        \'segment\': str(profile[\'segment\']), \'is_outlier\': outlier, \'product_categories\': product_names,\n        \'relative_spending\': {NAMES[name]: float(value) for name, value in top.items()},\n        \'campaign\': campaign, \'message\': message,\n        \'status\': \'Bản nháp gợi ý — sẵn sàng áp dụng\',\n        \'approval_checks\': [\n            \'Kiểm tra tồn kho các mặt hàng nổi bật\',\n            \'Xác nhận ngưỡng hóa đơn áp dụng voucher / chiết khấu\',\n            \'Gửi thông báo qua ứng dụng / SMS khách hàng thân thiết\',\n        ],\n    }\n\n\ndef render_results_tab(data):\n    """Hiển thị kết quả phân cụm, biểu đồ PCA, hồ sơ cụm và gợi ý khuyến mãi."""\n    folder = latest_result()\n    if folder is None:\n        st.info(\'Chưa có mô hình. Mở tab **Thu thập dữ liệu & Huấn luyện lại** để huấn luyện mô hình đầu tiên.\')\n        return\n    meta = json.loads((folder / \'metadata.json\').read_text(encoding=\'utf-8\'))\n    customers = pd.read_csv(folder / \'assignments.csv\')\n    segments = pd.read_csv(folder / \'segments.csv\')\n    medians = pd.read_csv(folder / \'medians.csv\').set_index(\'cluster\')\n    ratios = pd.read_csv(folder / \'relative_spending.csv\').set_index(\'cluster\')\n\n    k1, k2, k3, k4, k5 = st.columns(5)\n    k1.metric(\'Tổng khách hàng\', len(customers))\n    k2.metric(\'Số phân khúc\', meta[\'metrics\'][\'clusters\'])\n    k3.metric(\'Thuật toán chính\', meta[\'algorithm\'])\n    k4.metric(\'Ngoại lai (IQR)\', int(customers.get(\'is_outlier\', pd.Series(dtype=bool)).sum()))\n    k5.metric(\'Silhouette Score\', f"{meta[\'metrics\'][\'silhouette\']:.3f}")\n\n    if data_fingerprint(data) != meta[\'data_sha256\']:\n        st.warning(\'Kho dữ liệu đã có thêm khách hàng mới chưa được đưa vào mô hình này. Vào tab **Thu thập dữ liệu & Huấn luyện lại** để cập nhật.\')\n\n    st.subheader(\'1. Cơ cấu phân khúc & Gợi ý khuyến mãi\')\n    left, right = st.columns([1, 2])\n    with left:\n        counts = customers.groupby(\'cluster\').size()\n        fig, ax = plt.subplots(figsize=(4.2, 3.2))\n        ax.pie(counts, labels=[f\'Nhóm {c}\' if c != -1 else \'Nhiễu\' for c in counts.index],\n               autopct=\'%1.0f%%\', colors=COLORS[:len(counts)], wedgeprops={\'width\': .38}, startangle=90)\n        ax.set_title(\'Tỷ lệ khách hàng theo cụm\')\n        st.pyplot(fig)\n        plt.close(fig)\n    with right:\n        for profile in segments.itertuples():\n            cluster = int(profile.cluster)\n            title = \'Điểm nhiễu (DBSCAN)\' if cluster == -1 else str(profile.segment)\n            body = f\'<b>Khuyến mãi:</b> {escape(str(profile.suggestion))}\'\n            st.markdown(\n                f\'<div style="padding:14px 16px;border:1px solid #cbd5e1;border-top:4px solid #0d9488;border-radius:10px;margin-bottom:10px;">\'\n                f\'<h4 style="margin:0 0 6px 0;">{escape(title)} · {int(profile.customers)} khách hàng</h4><div>{body}</div></div>\',\n                unsafe_allow_html=True,\n            )\n\n    st.subheader(\'2. Trực quan hóa phân cụm (PCA 2D) & Mức chi tiêu tương đối\')\n    c_pca, c_heat = st.columns(2)\n    with c_pca:\n        fig, ax = plt.subplots(figsize=(6.5, 4.2))\n        for label, group in customers.groupby(\'cluster\'):\n            core = group[~group[\'is_outlier\']] if \'is_outlier\' in group else group\n            ax.scatter(core.PC1, core.PC2, s=20, alpha=.65, label=f\'Nhóm {label}\' if label != -1 else \'Nhiễu\')\n        if \'is_outlier\' in customers and customers[\'is_outlier\'].any():\n            out = customers[customers.is_outlier]\n            ax.scatter(out.PC1, out.PC2, s=36, facecolors=\'none\', edgecolors=\'#111827\', linewidths=1.0, label=\'Ngoại lai (IQR)\')\n        ax.set(xlabel=\'PC1\', ylabel=\'PC2\', title=\'Phân bố cụm trên không gian PCA 2D\')\n        ax.legend(fontsize=8)\n        fig.tight_layout()\n        st.pyplot(fig)\n        plt.close(fig)\n        st.caption(f"PCA 2D giữ lại {sum(meta[\'pca_explained_variance\']):.1%} phương sai (chỉ dùng để vẽ hình; phân cụm dùng đủ 6 nhóm hàng).")\n    with c_heat:\n        st.write(\'Chi tiêu trung bình của từng cụm so với mức chung (1.0 = bằng trung bình):\')\n        st.dataframe(ratios[FEATURES].style.format(\'{:.2f}\').background_gradient(cmap=\'YlGnBu\'), width=\'stretch\')\n        st.write(\'Trung vị chi tiêu trên đơn vị gốc:\')\n        st.dataframe(medians[FEATURES].style.format(\'{:,.0f}\'), width=\'stretch\')\n\n    st.divider()\n    st.subheader(\'3. Tra cứu từng khách hàng & Xuất đề xuất khuyến mãi\')\n    options = customers.row_id.astype(int).tolist()\n    customer_id = st.selectbox(\'Chọn mã khách hàng\', options, format_func=lambda v: f\'KH-{v:04d}\', key=\'demo_customer\')\n    customer = customers.loc[customers.row_id == customer_id].iloc[0]\n    cluster = int(customer.cluster)\n    profile = segments.loc[segments.cluster == cluster].iloc[0]\n    is_outlier = bool(customer.get(\'is_outlier\', False))\n\n    a, b, c = st.columns(3)\n    a.metric(\'Tổng chi tiêu năm\', f\'{customer[FEATURES].sum():,.0f}\')\n    b.metric(\'Phân khúc\', str(profile.segment) + (\' (Ngoại lai)\' if is_outlier else \'\'))\n    c.metric(\'Kênh mua hàng\', \'Bán lẻ / Siêu thị\' if customer.Channel == 2 else \'Khách sạn / Nhà hàng / Café\')\n\n    chart_col, offer_col = st.columns(2)\n    with chart_col:\n        fig, ax = plt.subplots(figsize=(6.5, 3.8))\n        positions = np.arange(len(FEATURES))\n        ax.barh(positions - .18, customer[FEATURES].to_numpy(dtype=float), height=.35, color=\'#2563eb\', label=\'Khách hàng\')\n        ax.barh(positions + .18, medians.loc[cluster, FEATURES].to_numpy(dtype=float), height=.35, color=\'#0d9488\', label=\'Trung vị nhóm\')\n        ax.set_yticks(positions, [NAMES[f] for f in FEATURES])\n        ax.set_xlabel(\'Chi tiêu năm\')\n        ax.set_title(\'So sánh chi tiêu khách hàng với trung vị nhóm\')\n        ax.legend()\n        fig.tight_layout()\n        st.pyplot(fig)\n        plt.close(fig)\n    with offer_col:\n        draft = campaign_draft(customer, profile, ratios, folder.name, meta.get(\'eligible\', True))\n        if draft is None:\n            st.info(\'Khách hàng thuộc điểm nhiễu, cần xem xét thủ công.\')\n        else:\n            if st.button(\'Tạo đề xuất khuyến mãi\', type=\'primary\', key=\'generate_offer\'):\n                st.session_state[\'offer_draft\'] = draft\n            saved = st.session_state.get(\'offer_draft\')\n            if saved and saved[\'run_id\'] == folder.name and saved[\'customer_ref\'] == draft[\'customer_ref\']:\n                with st.container(border=True):\n                    st.markdown(f"**{saved[\'campaign\']}**")\n                    st.info(saved[\'message\'])\n                    for name, ratio in saved[\'relative_spending\'].items():\n                        st.write(f\'• **{name}**: cao gấp **{ratio:.2f} lần** trung bình chung\')\n                    st.download_button(\n                        \'Tải phiếu đề xuất (JSON)\',\n                        json.dumps(saved, ensure_ascii=False, indent=2).encode(\'utf-8\'),\n                        f"{saved[\'customer_ref\']}_offer.json", \'application/json\',\n                    )\n            else:\n                st.write(f"**Gợi ý mặc định:** {profile.suggestion}")\n\n    with st.expander(\'Danh sách toàn bộ khách hàng & Tải kết quả CSV\'):\n        cluster_filter = st.multiselect(\'Lọc theo nhóm\', sorted(customers.cluster.unique().tolist()))\n        visible = customers[customers.cluster.isin(cluster_filter)] if cluster_filter else customers\n        st.dataframe(visible, width=\'stretch\')\n        st.download_button(\'Tải kết quả phân khúc (CSV)\', visible.to_csv(index=False).encode(\'utf-8-sig\'), \'customer_segments.csv\', \'text/csv\')\n', 'app.py': 'import json\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom scipy.cluster.hierarchy import linkage, dendrogram\nfrom sklearn.neighbors import NearestNeighbors\nimport streamlit as st\n\nfrom pipeline import (\n    FEATURES, COLUMNS, CATEGORY_VI, train, validate, search,\n    latest_run, predict_segments, retrain_due,\n)\nfrom storage import initialize, load_data, add_batch, history\nfrom experience import render_results_tab\n\nst.set_page_config(page_title=\'Phân khúc khách hàng siêu thị\', layout=\'wide\')\nst.title(\'Phân khúc khách hàng siêu thị để gợi ý khuyến mãi\')\nst.caption(\'Bài toán Phân cụm · Bộ dữ liệu Wholesale Customers (UCI, 440 dòng) · K-Means / Hierarchical / DBSCAN\')\ninitialize()\ndata = load_data()\n\n# ---- Thanh bên: Trạng thái kho dữ liệu & Chính sách huấn luyện định kỳ ----\nst.sidebar.metric(\'Khách hàng trong kho\', len(data))\nst.sidebar.subheader(\'Huấn luyện định kỳ\')\nauto_retrain = st.sidebar.checkbox(\'Tự huấn luyện lại khi đến hạn\', value=True, key=\'policy_auto\')\nmin_new = st.sidebar.number_input(\'Khi có ≥ N khách hàng mới\', min_value=1, value=10, step=1, key=\'policy_min_new\')\nmax_days = st.sidebar.number_input(\'hoặc sau X ngày có dữ liệu mới\', min_value=1, value=7, step=1, key=\'policy_max_days\')\ndue, reason = retrain_due(len(data), min_new=int(min_new), max_days=int(max_days))\nfolder_now = latest_run()\nif folder_now is not None:\n    last_meta = json.loads((folder_now / \'metadata.json\').read_text(encoding=\'utf-8\'))\n    st.sidebar.caption(f"Phiên hiện tại: {last_meta[\'algorithm\']} {last_meta[\'params\']} ({last_meta[\'scaler_type\']}) · {last_meta[\'rows\']} KH")\nst.sidebar.caption((\'⏰ Đến hạn: \' if due else \'✅ \') + reason)\nif due and auto_retrain and not st.session_state.get(\'auto_train_failed\'):\n    try:\n        with st.spinner(f\'Đang tự huấn luyện lại: {reason}\'):\n            train(data)\n        st.toast(\'Đã cập nhật mô hình phân khúc mới.\')\n    except ValueError as error:\n        st.session_state[\'auto_train_failed\'] = True\n        st.error(f\'Huấn luyện tự động thất bại: {error}\')\n\ntabs = st.tabs([\n    \'1. Dữ liệu & Khám phá (EDA)\',\n    \'2. Tiền xử lý & So sánh 3 thuật toán\',\n    \'3. Kết quả phân khúc & Khuyến mãi\',\n    \'4. Nhập liệu mới & Huấn luyện lại\',\n])\n\n\n# ==============================================================================\n# TAB 1: TỔNG QUAN & KHÁM PHÁ DỮ LIỆU (EDA)\n# ==============================================================================\nwith tabs[0]:\n    st.subheader(\'Tổng quan bộ dữ liệu Wholesale Customers (UCI)\')\n    st.write(\'Mỗi dòng ghi nhận chi tiêu hàng năm của một khách hàng theo 6 nhóm mặt hàng: \'\n             \'**Tươi sống (`Fresh`), Sữa (`Milk`), Tạp hóa (`Grocery`), Đông lạnh (`Frozen`), \'\n             \'Chất tẩy rửa & giấy (`Detergents_Paper`), Thực phẩm chế biến (`Delicassen`)**.\')\n    c1, c2, c3, c4 = st.columns(4)\n    c1.metric(\'Số dòng\', len(data))\n    c2.metric(\'Số cột\', data.shape[1])\n    c3.metric(\'Giá trị thiếu (Missing)\', int(data.isna().sum().sum()))\n    c4.metric(\'Kênh bán lẻ (Channel=2)\', int((data.Channel == 2).sum()))\n\n    st.subheader(\'Phát hiện ngoại lai (Outliers) bằng quy tắc IQR — Boxplot\')\n    fig, ax = plt.subplots(figsize=(11, 4))\n    ax.boxplot([data[col] for col in FEATURES], tick_labels=[CATEGORY_VI[c] for c in FEATURES], orientation=\'vertical\')\n    ax.set_ylabel(\'Chi tiêu năm (đơn vị gốc)\')\n    ax.set_title(\'Biểu đồ Boxplot 6 nhóm hàng trước khi loại bỏ ngoại lai\')\n    plt.xticks(rotation=15)\n    fig.tight_layout()\n    st.pyplot(fig)\n    plt.close(fig)\n    st.caption(\'Các dấu tròn phía trên râu Boxplot là giá trị ngoại lai (vượt quá Q3 + 1.5×IQR) cần xử lý ở bước tiền xử lý để không làm lệch tâm cụm.\')\n\n    with st.expander(\'Xem bảng dữ liệu gốc & Thống kê mô tả\'):\n        st.dataframe(data, width=\'stretch\')\n        st.dataframe(data[FEATURES].describe().T, width=\'stretch\')\n\n\n# ==============================================================================\n# TAB 2: TIỀN XỬ LÝ & SO SÁNH 3 THUẬT TOÁN\n# ==============================================================================\nwith tabs[1]:\n    st.subheader(\'Cấu hình tiền xử lý & Khảo sát trực tiếp\')\n    col_p1, col_p2, col_p3 = st.columns(3)\n    outlier_opt = col_p1.selectbox(\n        \'Loại bỏ ngoại lai\',\n        [\'iqr\', \'iqr_log\', \'none\'],\n        format_func=lambda x: {\'iqr\': \'IQR trực tiếp (Chuẩn)\', \'iqr_log\': \'IQR trên thang Log1p\', \'none\': \'Không loại ngoại lai\'}[x],\n    )\n    scaler_opt = col_p2.selectbox(\n        \'Chuẩn hóa dữ liệu\',\n        [\'standard\', \'minmax\'],\n        format_func=lambda x: {\'standard\': \'StandardScaler (Z-score)\', \'minmax\': \'MinMaxScaler [0, 1]\'}[x],\n    )\n    use_log_opt = col_p3.checkbox(\'Biến đổi Log1p trước chuẩn hóa\', value=False)\n\n    res = search(data, scaler_type=scaler_opt, outlier_method=outlier_opt, use_log=use_log_opt)\n    st.info(f\'Sau tiền xử lý: giữ **{int((~res.outlier).sum())}/{len(data)}** khách hàng đạt chuẩn để huấn luyện \'\n            f\'(đã tách riêng **{int(res.outlier.sum())}** ngoại lai). Phương pháp Khuỷu tay gợi ý **k = {res.elbow}**.\')\n\n    st.subheader(\'1. Đánh giá K-Means & Hierarchical: Elbow Method và Silhouette Score\')\n    km = res.table[res.table.algorithm == \'KMeans\'].copy()\n    hc = res.table[res.table.algorithm == \'Hierarchical\'].copy()\n    km[\'k\'] = km.params.map(lambda x: json.loads(x)[\'k\'])\n    hc[\'k\'] = hc.params.map(lambda x: json.loads(x)[\'k\'])\n\n    fig, axes = plt.subplots(1, 2, figsize=(12, 4))\n    axes[0].plot(km.k, km.inertia, \'o-\', color=\'#2563eb\')\n    axes[0].axvline(res.elbow, color=\'#dc2626\', linestyle=\'--\', label=f\'Điểm khuỷu gợi ý k = {res.elbow}\')\n    axes[0].set(xlabel=\'Số cụm (k)\', ylabel=\'Inertia (WCSS)\', title=\'Phương pháp Khuỷu tay (Elbow Method)\')\n    axes[0].legend()\n\n    axes[1].plot(km.k, km.silhouette, \'o-\', label=\'K-Means\')\n    axes[1].plot(hc.k, hc.silhouette, \'s-\', label=\'Hierarchical (Ward)\')\n    axes[1].set(xlabel=\'Số cụm (k)\', ylabel=\'Silhouette Score (càng cao càng tốt)\', title=\'So sánh Silhouette Score theo k\')\n    axes[1].legend()\n    fig.tight_layout()\n    st.pyplot(fig)\n    plt.close(fig)\n\n    st.subheader(\'2. Trực quan hóa Hierarchical (Dendrogram) & DBSCAN (k-distance)\')\n    c_den, c_kd = st.columns(2)\n    with c_den:\n        fig, ax = plt.subplots(figsize=(6, 4))\n        dendrogram(linkage(res.X_fit, method=\'ward\'), truncate_mode=\'lastp\', p=20, ax=ax)\n        ax.set(title=\'Dendrogram (Hierarchical Ward, rút gọn 20 nhánh)\', xlabel=\'Nhóm gộp\', ylabel=\'Khoảng cách\')\n        fig.tight_layout()\n        st.pyplot(fig)\n        plt.close(fig)\n    with c_kd:\n        fig, ax = plt.subplots(figsize=(6, 4))\n        dists = np.sort(NearestNeighbors(n_neighbors=4).fit(res.X_fit).kneighbors(res.X_fit)[0][:, -1])\n        ax.plot(dists, color=\'#0d9488\')\n        ax.set(title=\'Biểu đồ k-distance (chọn eps cho DBSCAN)\', xlabel=\'Khách hàng sắp xếp tăng dần\', ylabel=\'Khoảng cách láng giềng thứ 4\')\n        fig.tight_layout()\n        st.pyplot(fig)\n        plt.close(fig)\n\n    st.subheader(\'3. Bảng so sánh 3 thuật toán: K-Means · Hierarchical · DBSCAN\')\n    best_show = res.best[[\'algorithm\', \'params\', \'clusters\', \'noise_rate\', \'silhouette\', \'davies_bouldin\', \'calinski_harabasz\', \'seconds\', \'eligible\']].copy()\n    best_show.columns = [\'Thuật toán\', \'Tham số tốt nhất\', \'Số cụm\', \'Tỷ lệ nhiễu\', \'Silhouette\', \'Davies-Bouldin\', \'Calinski-Harabasz\', \'Thời gian (s)\', \'Đạt tiêu chí\']\n    st.dataframe(best_show, width=\'stretch\')\n\n    if st.button(\'Áp dụng cấu hình này & Huấn luyện lưu phiên\', type=\'primary\', key=\'apply_tab2\'):\n        with st.spinner(\'Đang huấn luyện và lưu kết quả...\'):\n            folder = train(data, scaler_type=scaler_opt, outlier_method=outlier_opt, use_log=use_log_opt)\n        st.success(f\'Đã huấn luyện và lưu phiên mới: {folder.name}. Mở Tab 3 để xem chi tiết phân khúc!\')\n\n    with st.expander(\'Xem toàn bộ bảng khảo sát tham số (K-Means, Hierarchical, DBSCAN)\'):\n        st.dataframe(res.table, width=\'stretch\')\n\n\n# ==============================================================================\n# TAB 3: KẾT QUẢ PHÂN KHÚC & GỢI Ý KHUYẾN MÃI\n# ==============================================================================\nwith tabs[2]:\n    render_results_tab(data)\n\n\n# ==============================================================================\n# TAB 4: THU THẬP DỮ LIỆU MỚI & HUẤN LUYỆN LẠI ĐỊNH KỲ\n# ==============================================================================\nwith tabs[3]:\n    col_form, col_train = st.columns([1.2, 1])\n\n    with col_form:\n        st.subheader(\'Nhập khách hàng mới (gán phân khúc ngay lập tức)\')\n        presets = {\n            \'Khách mua sắm tiết kiệm\': dict(Channel=1, Region=3, Fresh=3000., Milk=1200., Grocery=1500., Frozen=1000., Detergents_Paper=400., Delicassen=500.),\n            \'Cửa hàng tạp hóa / gia dụng\': dict(Channel=2, Region=1, Fresh=4000., Milk=8500., Grocery=12000., Frozen=1200., Detergents_Paper=4500., Delicassen=800.),\n            \'Nhà hàng / Tươi sống & Đông lạnh\': dict(Channel=1, Region=3, Fresh=18000., Milk=2000., Grocery=2500., Frozen=5500., Detergents_Paper=500., Delicassen=1200.),\n        }\n        p_cols = st.columns(3)\n        for idx, (label, vals) in enumerate(presets.items()):\n            if p_cols[idx].button(f\'Mẫu: {label}\', key=f\'preset_{idx}\'):\n                st.session_state[\'in_channel\'] = vals[\'Channel\']\n                st.session_state[\'in_region\'] = vals[\'Region\']\n                for feat in FEATURES:\n                    st.session_state[f\'in_{feat}\'] = vals[feat]\n\n        with st.form(\'customer\'):\n            c_a, c_b = st.columns(2)\n            channel = c_a.selectbox(\'Channel\', [1, 2], key=\'in_channel\',\n                                    format_func=lambda x: {1: \'1 — Khách sạn / Nhà hàng\', 2: \'2 — Bán lẻ / Siêu thị\'}[x])\n            region = c_b.selectbox(\'Region\', [1, 2, 3], key=\'in_region\',\n                                   format_func=lambda x: {1: \'1 — Lisbon\', 2: \'2 — Oporto\', 3: \'3 — Khác\'}[x])\n            f_cols = st.columns(2)\n            values = {}\n            for i, col in enumerate(FEATURES):\n                values[col] = f_cols[i % 2].number_input(\n                    f\'{CATEGORY_VI[col]} ({col})\', min_value=0., step=100.,\n                    value=float(st.session_state.get(f\'in_{col}\', 0.)), key=f\'in_{col}\',\n                )\n            confirmed = st.checkbox(\'Đã kiểm tra đây là khách hàng mới và số liệu hợp lệ\')\n            submit = st.form_submit_button(\'Lưu & Dự đoán phân khúc\', type=\'primary\')\n\n        if submit:\n            if not confirmed:\n                st.warning(\'Cần đánh dấu xác nhận trước khi lưu.\')\n            else:\n                try:\n                    new = pd.DataFrame([dict(Channel=channel, Region=region, **values)])\n                    count = add_batch(new, \'Nhập từ biểu mẫu\')\n                    st.session_state[\'save_notice\'] = f\'Đã lưu {count} khách hàng mới vào CSDL.\' if count else \'Bản ghi giống hệt đã tồn tại; không thêm trùng.\'\n                    st.session_state[\'save_prediction\'] = predict_segments(new) if count else None\n                    st.rerun()\n                except ValueError as error:\n                    st.error(str(error))\n\n        if \'save_notice\' in st.session_state:\n            st.success(st.session_state.pop(\'save_notice\'))\n        if st.session_state.get(\'save_prediction\') is not None:\n            pred = st.session_state.pop(\'save_prediction\')\n            with st.container(border=True):\n                st.markdown(\'**Kết quả phân khúc & khuyến mãi cho khách hàng vừa nhập:**\')\n                for row in pred.itertuples():\n                    st.write(f\'• Phân khúc: **{row.segment}**\' + (\' · ⚠️ Ngoại lai\' if row.is_outlier else \'\'))\n                    st.info(row.suggestion)\n\n        st.divider()\n        st.subheader(\'Nhập hàng loạt từ CSV\')\n        st.download_button(\'Tải file mẫu CSV\', pd.DataFrame(columns=COLUMNS).to_csv(index=False).encode(\'utf-8-sig\'), \'customer_template.csv\', \'text/csv\')\n        uploaded = st.file_uploader(\'Chọn file CSV (đủ 8 cột)\', type=\'csv\')\n        if uploaded is not None:\n            try:\n                incoming = validate(pd.read_csv(uploaded))\n                st.dataframe(incoming.head(10))\n                agree = st.checkbox(\'Xác nhận bổ sung tập khách hàng từ CSV này\')\n                if st.button(\'Lưu CSV vào kho\', disabled=not agree):\n                    count = add_batch(incoming, uploaded.name)\n                    st.session_state[\'save_notice\'] = f\'Đã lưu {count} khách hàng.\' if count else \'File này đã được nhập trước đó.\'\n                    st.session_state[\'save_prediction\'] = predict_segments(incoming.head(3)) if count else None\n                    st.rerun()\n            except (ValueError, pd.errors.ParserError, UnicodeError) as error:\n                st.error(str(error))\n\n    with col_train:\n        st.subheader(\'Huấn luyện lại mô hình\')\n        st.write(\'Chọn cấu hình và bấm nút để huấn luyện lại trên toàn bộ dữ liệu hiện có trong SQLite:\')\n        alg_opt = st.selectbox(\'Thuật toán chính\', [\'Auto\', \'KMeans\', \'Hierarchical\', \'DBSCAN\'])\n        sc_opt = st.selectbox(\'Bộ chuẩn hóa\', [\'standard\', \'minmax\'], format_func=lambda x: \'StandardScaler\' if x == \'standard\' else \'MinMaxScaler\', key=\'tr_sc\')\n        out_opt = st.selectbox(\'Xử lý ngoại lai\', [\'iqr\', \'iqr_log\', \'none\'], format_func=lambda x: {\'iqr\': \'IQR trực tiếp\', \'iqr_log\': \'IQR (Log1p)\', \'none\': \'Không loại\'}[x], key=\'tr_out\')\n        if st.button(\'Huấn luyện lại ngay\', type=\'primary\', key=\'btn_retrain\'):\n            try:\n                with st.spinner(\'Đang huấn luyện lại mô hình...\'):\n                    folder = train(load_data(), scaler_type=sc_opt, outlier_method=out_opt, algorithm=alg_opt)\n                st.session_state[\'auto_train_failed\'] = False\n                st.success(f\'Huấn luyện xong phiên {folder.name}\')\n            except ValueError as error:\n                st.error(str(error))\n        st.caption(\'Ngoài giao diện, có thể đặt lịch tự động chạy `python retrain.py --if-due` trên Windows Task Scheduler.\')\n\n        st.subheader(\'Lịch sử nhập dữ liệu\')\n        st.dataframe(history(), width=\'stretch\')\n', 'retrain.py': '"""Chạy tay hoặc bằng Windows Task Scheduler / cron. Không cần mở Streamlit.\n\nVí dụ:\n  python retrain.py\n  python retrain.py --scaler minmax --outlier iqr_log\n  python retrain.py --if-due\n"""\nimport argparse\nfrom pipeline import train, retrain_due\nfrom storage import initialize, load_data\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--algorithm\', choices=[\'Auto\', \'KMeans\', \'Hierarchical\', \'DBSCAN\'], default=\'Auto\')\n    parser.add_argument(\'--scaler\', choices=[\'standard\', \'minmax\'], default=\'standard\')\n    parser.add_argument(\'--outlier\', choices=[\'iqr\', \'iqr_log\', \'none\'], default=\'iqr\')\n    parser.add_argument(\'--use-log\', action=\'store_true\', help=\'Biến đổi log1p trước khi chuẩn hóa.\')\n    parser.add_argument(\'--allow-ineligible\', action=\'store_true\', help=\'Cho phép lưu cấu hình chưa đạt tiêu chí.\')\n    parser.add_argument(\'--if-due\', action=\'store_true\', help=\'Chỉ huấn luyện khi đủ khách mới hoặc quá hạn ngày.\')\n    parser.add_argument(\'--min-new\', type=int, default=10, help=\'Ngưỡng số khách hàng mới (dùng với --if-due).\')\n    parser.add_argument(\'--max-days\', type=int, default=7, help=\'Ngưỡng số ngày (dùng với --if-due).\')\n    args = parser.parse_args()\n    initialize()\n    data = load_data()\n    if args.if_due:\n        due, reason = retrain_due(len(data), min_new=args.min_new, max_days=args.max_days)\n        print(reason)\n        if not due:\n            return\n    folder = train(\n        data, scaler_type=args.scaler, outlier_method=args.outlier,\n        use_log=args.use_log, algorithm=args.algorithm, allow_ineligible=args.allow_ineligible,\n    )\n    print(folder)\n    print((folder / \'metadata.json\').read_text(encoding=\'utf-8\'))\n\n\nif __name__ == \'__main__\':\n    main()\n'}
for name, text in PROJECT_FILES.items():
    (PROJECT_DIR / name).write_text(text, encoding='utf-8')
sys.path.insert(0, str(PROJECT_DIR))
import importlib, pipeline
importlib.reload(pipeline)
from pipeline import *

raw = pd.read_csv(PROJECT_DIR / 'data' / 'wholesale_customers.csv')
data = validate(raw)
print('Kích thước dữ liệu:', data.shape, '| Giá trị thiếu:', raw.isna().sum().sum(), '| Dòng trùng:', raw.duplicated().sum())
display(data.head())
display(data[FEATURES].describe().T)

## 2. Tiền xử lý: Phát hiện ngoại lai (IQR) & Chuẩn hóa dữ liệu
1. **Phát hiện & loại bỏ ngoại lai (Outliers):** Dùng quy tắc **IQR** ($[Q_1 - 1.5\times IQR,\; Q_3 + 1.5\times IQR]$). Vì các thuật toán dựa trên khoảng cách rất nhạy với giá trị cực đại, ta loại ngoại lai khỏi tập huấn luyện (nhưng vẫn gắn cờ để chăm sóc riêng ở bước khuyến mãi, không làm mất dữ liệu khách hàng).
2. **Chuẩn hóa dữ liệu:** Đưa 6 thuộc tính về cùng thang đo bằng **`StandardScaler`** (Z-score) hoặc **`MinMaxScaler`** ($[0, 1]$). Bắt buộc phải chuẩn hóa vì biên độ chi tiêu giữa các nhóm hàng chênh lệch rất lớn.

In [ ]:
# Trực quan hóa Boxplot trước và sau khi loại bỏ ngoại lai bằng IQR
outlier_mask, bounds = detect_outliers_iqr(data, method='iqr')
clean_data = data.loc[~outlier_mask].reset_index(drop=True)
print(f'Tổng số khách hàng: {len(data)} | Ngoại lai (IQR): {outlier_mask.sum()} | Tập sạch để huấn luyện: {len(clean_data)}')

fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
axes[0].boxplot([data[c] for c in FEATURES], tick_labels=FEATURES)
axes[0].set_title(f'Trước khi loại ngoại lai ({len(data)} dòng)'); axes[0].tick_params(axis='x', rotation=25)
axes[1].boxplot([clean_data[c] for c in FEATURES], tick_labels=FEATURES)
axes[1].set_title(f'Sau khi loại ngoại lai bằng IQR ({len(clean_data)} dòng)'); axes[1].tick_params(axis='x', rotation=25)
plt.tight_layout(); plt.savefig(FIGURES / 'boxplot_outliers.png', dpi=140); plt.show()

# Đối chiếu 2 bộ chuẩn hóa theo đề cương: StandardScaler vs MinMaxScaler
scaler_comp = compare_scalers(data, outlier_method='iqr')
display(scaler_comp[['scaler', 'algorithm', 'params', 'clusters', 'noise_rate', 'silhouette', 'davies_bouldin', 'eligible']].round(3))

## 3. Huấn luyện & Đánh giá 3 thuật toán (K-Means · Hierarchical · DBSCAN)
- **K-Means:** Khảo sát $k = 2 \dots 8$, dùng **Elbow Method** (Inertia/WCSS) và **Silhouette Score** để chọn $k$.
- **Hierarchical Clustering (Agglomerative - Ward):** Vẽ **Dendrogram** và khảo sát $k = 2 \dots 8$.
- **DBSCAN:** Dựa vào biểu đồ **k-distance** để chọn `eps` và khảo sát lưới `(eps, min_samples)`.
- Lưu ý: Với bài toán gợi ý khuyến mãi, chọn $k \ge 3$ để tránh chỉ tách lại 2 kênh `Channel` có sẵn.

In [ ]:
res = search(data, scaler_type='standard', outlier_method='iqr', use_log=False)
table = res.table.copy()
print('Điểm khuỷu (Elbow Method) gợi ý k =', res.elbow)

km = table[table.algorithm == 'KMeans'].assign(k=lambda d: d.params.map(lambda x: json.loads(x)['k']))
hc = table[table.algorithm == 'Hierarchical'].assign(k=lambda d: d.params.map(lambda x: json.loads(x)['k']))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(km.k, km.inertia, 'o-', color='#2563eb')
axes[0].axvline(res.elbow, color='#dc2626', ls='--', label=f'Elbow gợi ý k = {res.elbow}')
axes[0].set(xlabel='Số cụm (k)', ylabel='Inertia (WCSS)', title='Phương pháp Khuỷu tay (Elbow Method - KMeans)')
axes[0].legend()

axes[1].plot(km.k, km.silhouette, 'o-', label='K-Means')
axes[1].plot(hc.k, hc.silhouette, 's-', label='Hierarchical (Ward)')
axes[1].set(xlabel='Số cụm (k)', ylabel='Silhouette Score', title='So sánh Silhouette Score theo k')
axes[1].legend()
plt.tight_layout(); plt.savefig(FIGURES / 'elbow_and_silhouette.png', dpi=140); plt.show()

In [ ]:
# Biểu đồ Silhouette chi tiết từng mẫu cho K-Means k = 2..5
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_samples
Xf = res.X_fit
fig, axes = plt.subplots(1, 4, figsize=(20, 4.2))
for ax, k in zip(axes, [2, 3, 4, 5]):
    lab = KMeans(n_clusters=k, random_state=42, n_init=20).fit_predict(Xf)
    s = silhouette_samples(Xf, lab); y = 10
    for c in range(k):
        v = np.sort(s[lab == c])
        ax.fill_betweenx(np.arange(y, y + len(v)), 0, v, alpha=.75)
        ax.text(-0.08, y + len(v) / 2, str(c), fontsize=9)
        y += len(v) + 10
    ax.axvline(s.mean(), color='red', ls='--'); ax.set_yticks([]); ax.grid(False)
    ax.set(title=f'k = {k} (Sil = {s.mean():.3f})', xlabel='Silhouette')
plt.tight_layout(); plt.savefig(FIGURES / 'silhouette_per_k.png', dpi=140); plt.show()

# Dendrogram (Hierarchical) & k-distance (DBSCAN)
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.neighbors import NearestNeighbors
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
dendrogram(linkage(Xf, method='ward'), truncate_mode='lastp', p=25, ax=axes[0])
axes[0].set(title='Dendrogram (Hierarchical Ward, rút gọn 25 nhánh)', xlabel='Nhóm gộp', ylabel='Khoảng cách')
dists = np.sort(NearestNeighbors(n_neighbors=4).fit(Xf).kneighbors(Xf)[0][:, -1])
axes[1].plot(dists, color='#0d9488')
axes[1].set(title='Biểu đồ k-distance (DBSCAN, k=4)', xlabel='Khách hàng sắp xếp tăng dần', ylabel='Khoảng cách láng giềng thứ 4')
plt.tight_layout(); plt.savefig(FIGURES / 'dendrogram_and_kdistance.png', dpi=140); plt.show()

## 4. So sánh kết quả 3 thuật toán & Trực quan hóa PCA 2D

In [ ]:
display(res.best[['algorithm', 'params', 'clusters', 'noise_rate', 'silhouette', 'davies_bouldin', 'calinski_harabasz', 'seconds', 'eligible']].round(3))

pca = PCA(n_components=2).fit(res.X_fit)
coords = pca.transform(res.X)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, name in zip(axes, ['KMeans', 'Hierarchical', 'DBSCAN']):
    row = res.best[res.best.algorithm == name].iloc[0]
    lab = full_labels(res, res.candidates[int(row.candidate)][1])
    for c in sorted(set(lab)):
        m = (lab == c) & ~res.outlier
        ax.scatter(coords[m, 0], coords[m, 1], s=18, alpha=.7, label='Nhiễu' if c == -1 else f'Cụm {c}')
    ax.scatter(coords[res.outlier, 0], coords[res.outlier, 1], s=32, c='k', marker='x', alpha=.5, label='Ngoại lai (IQR)')
    ax.set_title(f"{name} {row.params}\nSilhouette = {row.silhouette:.3f}")
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIGURES / 'three_algorithms_pca.png', dpi=140); plt.show()
print(f'PCA 2D giải thích {pca.explained_variance_ratio_.sum():.1%} phương sai.')

## 5. Hồ sơ phân khúc khách hàng & Gợi ý khuyến mãi
Huấn luyện phiên chính thức, lưu vào `wholesale_app/artifacts/` (dùng chung cho giao diện Streamlit) và trích xuất bảng hồ sơ chi tiêu + gợi ý khuyến mãi cho từng phân khúc.

In [ ]:
folder = train(data, output_root=PROJECT_DIR / 'artifacts', scaler_type='standard', outlier_method='iqr', use_log=False)
metadata = json.loads((folder / 'metadata.json').read_text(encoding='utf-8'))
segments = pd.read_csv(folder / 'segments.csv')
relative = pd.read_csv(folder / 'relative_spending.csv').set_index('cluster')
medians = pd.read_csv(folder / 'medians.csv')

print('Phiên lưu:', folder.name, '| Thuật toán chọn:', metadata['algorithm'], metadata['params'])
pd.set_option('display.max_colwidth', 220)
display(segments[['cluster', 'customers', 'outliers_assigned', 'segment', 'suggestion', 'rationale']])

fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(relative[FEATURES], aspect='auto', cmap='YlGnBu'); ax.grid(False)
ax.set_xticks(range(6), [CATEGORY_VI[f] for f in FEATURES], rotation=25, ha='right')
ax.set_yticks(range(len(relative)), [f'Cụm {c}' for c in relative.index])
for i in range(relative.shape[0]):
    for j in range(6):
        ax.text(j, i, f'{relative[FEATURES].iloc[i, j]:.2f}', ha='center', va='center', fontsize=9)
ax.set_title('Mức chi tiêu tương đối của từng cụm (1.0 = bằng mức trung bình chung)')
fig.colorbar(im, ax=ax)
plt.tight_layout(); plt.savefig(FIGURES / 'segment_heatmap.png', dpi=140); plt.show()

# Thử nghiệm gán phân khúc cho khách hàng mới ngay lập tức
demo_new = pd.DataFrame([
    [2, 1, 4000, 8500, 12000, 1200, 4500, 800],
    [1, 3, 18000, 2000, 2500, 5500, 500, 1200],
], columns=COLUMNS)
display(predict_segments(demo_new, folder)[['Channel', 'cluster', 'segment', 'is_outlier', 'suggestion']])

## 6. Giao diện ứng dụng (Streamlit) & Đóng gói kết quả
Giao diện đáp ứng đủ 4 yêu cầu: thu thập dữ liệu mới (form + CSV), tiền xử lý, huấn luyện lại định kỳ và hiển thị kết quả phân khúc + khuyến mãi cho người dùng.

In [ ]:
RUN_STREAMLIT = 'google.colab' in sys.modules
if RUN_STREAMLIT:
    import subprocess, time
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'streamlit'], check=True)
    log_file = open(WORK / 'streamlit.log', 'w')
    server = subprocess.Popen(
        [sys.executable, '-m', 'streamlit', 'run', 'app.py', '--server.port', '8501',
         '--server.headless', 'true', '--server.enableCORS', 'false', '--server.enableXsrfProtection', 'false'],
        cwd=PROJECT_DIR, stdout=log_file, stderr=subprocess.STDOUT,
    )
    time.sleep(10)
    from google.colab import output
    output.serve_kernel_port_as_window(8501)
    print('Streamlit đang chạy tại cổng 8501. Nếu trình duyệt chặn pop-up, chạy: output.serve_kernel_port_as_iframe(8501)')
else:
    print('Chạy trên máy cá nhân: `python -m streamlit run app.py` trong thư mục', PROJECT_DIR)

shutil.copytree(FIGURES, folder / 'figures', dirs_exist_ok=True)
zip_path = shutil.make_archive(str(WORK / 'wholesale_results'), 'zip', folder)
print('Đã đóng gói toàn bộ kết quả & biểu đồ tại:', zip_path)